# p138 Copy List with Random Pointer 解析笔记

- Top100 序号：57
- 题目英文名：Copy List with Random Pointer
- 题目中文名：复制带随机指针的链表
- 题目链接：https://leetcode.cn/problems/copy-list-with-random-pointer/
- 题型：链表
- 难度：Medium
- 推荐优先级：🟡 中


## 1. 题目一句话总结

给定一个带 `next` 和 `random` 指针的链表，需要我们复制出一条值相同、连接关系也完全一致的新链表，而且新链表里的随机指针不能再指向旧链表中的节点。

这题的关键不是单纯复制节点，而是正确维护“旧节点到新节点”的映射关系。

## 2. 题目理解

### 2.1 题目要求

链表中的每个节点除了有 `next` 指针外，还有一个 `random` 指针，可以指向链表中的任意节点，也可以指向 `None`。

我们要构造一条全新的链表，使得：

- 新链表节点值与原链表一致
- `next` 关系一致
- `random` 关系一致
- 新链表中的任意指针都不能复用原链表节点

### 2.2 输入与输出

- 输入：原链表头节点 `head`
- 输出：深拷贝后的新链表头节点

### 2.3 关键约束

- 需要完成深拷贝，不能只复制值
- `random` 可能为 `None`
- `random` 也可能指回当前节点自己
- 需要尽量降低空间复杂度


## 3. 朴素思路

### 3.1 最直接的做法

最容易想到的方法是使用哈希表：

1. 第一次遍历，复制每个节点，并记录 `原节点 -> 新节点`
2. 第二次遍历，根据哈希表补齐每个新节点的 `next` 和 `random`

### 3.2 为什么还能继续优化

这个做法时间复杂度是 `O(n)`，已经很好，但空间复杂度是 `O(n)`。

如果我们能把复制节点直接插入原链表中，就能临时借用链表结构本身完成映射，而不用额外哈希表。

## 4. 核心算法思路

### 4.1 算法名称

- 链表节点穿插
- 原地映射

### 4.2 为什么想到这个算法

如果把每个复制节点直接插到原节点后面，就会形成这样的结构：

```text
原链表：A -> B -> C
穿插后：A -> A' -> B -> B' -> C -> C'
```

这样一来：

- `A'` 就是 `A.next`
- `B'` 就是 `B.next`
- 如果 `A.random = B`，那么 `A'.random` 就应该指向 `B'`，也就是 `A.random.next`

于是我们就不需要额外哈希表了。

### 4.3 三步走

1. 第一次遍历：复制每个节点，并把复制节点插到原节点后面
2. 第二次遍历：利用 `cur.random.next` 设置复制节点的 `random`
3. 第三次遍历：把新旧链表拆开，恢复原链表并返回新链表


## 5. 图解

设原链表如下：

```text
7 -> 13 -> 11
     ^     |
     |_____|
```

其中：

- `7.random = None`
- `13.random = 7`
- `11.random = 13`

### 5.1 第一步：复制并穿插

```text
7 -> 7' -> 13 -> 13' -> 11 -> 11'
```

这时每个原节点后面都紧跟着它的复制节点。

### 5.2 第二步：连接 random

如果当前原节点是 `cur`：

- 原节点随机指向 `cur.random`
- 复制节点是 `cur.next`
- 随机目标的复制节点就是 `cur.random.next`

所以：

```python
cur.next.random = cur.random.next if cur.random else None
```

### 5.3 第三步：拆分两条链

```text
旧链表：7 -> 13 -> 11
新链表：7' -> 13' -> 11'
```

这样既得到答案，也把原链表恢复回去了。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(eq=False)
class Node:
    val: int
    next: Node | None = None
    random: Node | None = None


class Solution:
    def copyRandomList(self, head: Node | None) -> Node | None:
        if head is None:
            return None

        cur = head
        while cur is not None:
            nxt = cur.next
            copy = Node(cur.val)
            cur.next = copy
            copy.next = nxt
            cur = nxt

        cur = head
        while cur is not None:
            copy = cur.next
            copy.random = cur.random.next if cur.random is not None else None
            cur = copy.next

        cur = head
        new_head = head.next
        while cur is not None:
            copy = cur.next
            nxt = copy.next

            cur.next = nxt
            copy.next = nxt.next if nxt is not None else None
            cur = nxt

        return new_head


## 6. 代码逐段讲解

### 6.1 穿插复制节点

第一次遍历时，把 `copy` 插到 `cur` 后面，这样复制节点和原节点天然配对。

### 6.2 设置 random

原节点 `cur` 的复制节点是 `cur.next`，而原随机目标 `cur.random` 的复制节点正好是 `cur.random.next`。

### 6.3 拆分链表

最后一次遍历要同时做两件事：

- 恢复原链表的 `next`
- 让复制链表的 `next` 串起来


## 7. 复杂度分析

- 时间复杂度：`O(n)`
- 空间复杂度：`O(1)`（不计输出链表本身）

原因是我们只做了三次线性遍历，没有额外使用与节点数成比例的辅助结构。

## 8. 易错点

1. 忘记处理空链表，导致 `head.next` 报错。
2. 设置 `random` 时直接写成 `copy.random = cur.random`，这样会把新链表指回旧链表。
3. 拆分时没有先保存 `nxt`，容易把后续链表断开。
4. 误以为空间复杂度不是 `O(1)`，其实输出链表本身不计入辅助空间。

## 9. 面试时怎么讲

可以这样概括：

> 这题最直接的方法是哈希表记录旧节点到新节点的映射，但我可以进一步优化到 `O(1)` 额外空间。  
> 做法是先把复制节点穿插到原节点后面，这样每个旧节点和新节点天然成对。  
> 然后利用 `cur.random.next` 给复制节点补上随机指针，最后再把两条链拆开。  
> 整体时间复杂度 `O(n)`，额外空间复杂度 `O(1)`。

## 10. 实际应用场景

这题对应的真实问题可以理解为：复制一个带有普通引用和跨节点引用的对象链。

### 10.1 具体业务案例：工作流节点复制

#### 业务背景

在审批系统或工作流系统里，一条流程链可能既有顺序节点，也有“跳转到某个节点”的额外引用。

#### 输入是什么

输入是一条流程节点链：

- `next` 表示正常执行顺序
- `random` 表示补偿节点、快捷跳转节点或回退节点

#### 算法介入点

当系统要复制一套现有流程模板时，不能只复制节点值，还要把节点之间的引用关系完整迁移过去。

#### 输出是什么

输出是一条全新的流程链，引用关系保持一致，但不与原模板共享节点对象。

#### 它解决了什么问题

它解决的是“如何在不污染原结构的前提下，完整复制带复杂引用关系的链式对象”。

In [ ]:
def build_random_list(spec: list[tuple[int, int | None]]) -> Node | None:
    if not spec:
        return None

    nodes = [Node(val) for val, _ in spec]
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]

    for node, (_, random_index) in zip(nodes, spec):
        node.random = nodes[random_index] if random_index is not None else None

    return nodes[0]


def serialize_random_list(head: Node | None) -> list[tuple[int, int | None]]:
    nodes = []
    index_map: dict[Node, int] = {}
    cur = head

    while cur is not None:
        index_map[cur] = len(nodes)
        nodes.append(cur)
        cur = cur.next

    result = []
    for node in nodes:
        random_index = index_map[node.random] if node.random is not None else None
        result.append((node.val, random_index))
    return result


sample = [(7, None), (13, 0), (11, 1), (10, 2), (1, 0)]
head = build_random_list(sample)
copied = Solution().copyRandomList(head)

print('原链表序列化结果:', serialize_random_list(head))
print('复制链表序列化结果:', serialize_random_list(copied))
print('头节点是否复用同一对象:', head is copied)


## 11. Demo 输出说明

- 序列化结果相同，说明值和 `random` 指向关系都复制正确。
- `head is copied` 为 `False`，说明返回的是新链表，而不是原链表。

## 12. 扩展题

- `p133` Clone Graph
- `p138` Copy List with Random Pointer
- 各类“深拷贝复杂对象结构”的设计题

这些题可以放在一起理解：核心都是如何维护“旧对象到新对象”的映射。

## 13. 一句话复盘

> 这题最重要的不是复制值，而是想办法用最省空间的方式，维护好旧节点和新节点之间的一一对应关系。